# ITS Digital Twin: Perception Pipeline
This notebook downloads the AIC22 CityFlow dataset, extracts vehicle trajectories, projects them to 3D world coordinates, and exports the data for the Three.js frontend.

In [ ]:
!pip install -q ultralytics supervision opencv-python-headless kaggle scipy numpy

In [ ]:
import os
from google.colab import userdata
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Kaggle credentials loaded from Colab Secrets.')
except Exception:
    import getpass
    print('Please enter your Kaggle credentials.')
    os.environ['KAGGLE_USERNAME'] = getpass.getpass('KAGGLE_USERNAME: ')
    os.environ['KAGGLE_KEY'] = getpass.getpass('KAGGLE_KEY: ')

!kaggle datasets download -d shanthinisampath1/cityflow-multi-camera-tracking-dataset --unzip -p ./cityflow

In [ ]:
import numpy as np
calib_path = './cityflow/train/S01/c001/calibration.txt'
with open(calib_path, 'r') as f:
    h_values = list(map(float, f.readline().strip().split()))
    H = np.array(h_values).reshape(3, 3)

print('Homography Matrix (H):')
print(H)

def project_to_ground(u, v, H):
    point_2d = np.array([u, v, 1.0])
    point_3d = H @ point_2d
    point_3d = point_3d / point_3d[2]
    return point_3d[0], point_3d[1]

In [ ]:
import cv2
import json
from collections import defaultdict, deque
from ultralytics import YOLO

model = YOLO('yolov8x.pt')
video_path = './cityflow/train/S01/c001/vdo.avi'
output_video_path = 'feed_raw.mp4'
output_json_path = 'scene_data.json'

cap = cv2.VideoCapture(video_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

track_history = defaultdict(lambda: deque(maxlen=5))
last_state = {}
frames_data = {}
frame_idx = 1
world_bounds = {'minX': float('inf'), 'maxX': float('-inf'), 'minZ': float('inf'), 'maxZ': float('-inf')}

results = model.track(source=video_path, persist=True, stream=True, classes=[2, 3, 5, 7])
for r in results:
    frame = r.orig_img
    frame_data = []
    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy().astype(int)
        class_ids = r.boxes.cls.cpu().numpy().astype(int)
        for box, track_id, class_id in zip(boxes, track_ids, class_ids):
            x1, y1, x2, y2 = box
            class_name = model.names[class_id]
            u, v = (x1 + x2) / 2, y2
            X_w, Z_w = project_to_ground(u, v, H)
            world_bounds['minX'] = min(world_bounds['minX'], float(X_w))
            world_bounds['maxX'] = max(world_bounds['maxX'], float(X_w))
            world_bounds['minZ'] = min(world_bounds['minZ'], float(Z_w))
            world_bounds['maxZ'] = max(world_bounds['maxZ'], float(Z_w))
            
            track_history[track_id].append((X_w, Z_w))
            if len(track_history[track_id]) >= 3:
                hist_arr = np.array(track_history[track_id])
                smooth_X, smooth_Z = np.median(hist_arr[:, 0]), np.median(hist_arr[:, 1])
            else:
                smooth_X, smooth_Z = X_w, Z_w
                
            speed_kmh, yaw = 0.0, 0.0
            if track_id in last_state:
                prev_X, prev_Z, prev_frame = last_state[track_id]
                dt = (frame_idx - prev_frame) / fps
                dx, dz = smooth_X - prev_X, smooth_Z - prev_Z
                dist = np.sqrt(dx**2 + dz**2)
                if dt > 0: speed_kmh = (dist / dt) * 3.6
                yaw = np.arctan2(dz, dx)
            last_state[track_id] = (smooth_X, smooth_Z, frame_idx)
            
            frame_data.append({'id': int(track_id), 'class_name': class_name, 'x': float(smooth_X), 'z': float(smooth_Z), 'yaw': float(yaw), 'speed_kmh': float(speed_kmh), 'bbox': [float(x1), float(y1), float(x2), float(y2)]})
    frames_data[str(frame_idx)] = frame_data
    out.write(frame)
    frame_idx += 1

out.release()
cap.release()

!ffmpeg -y -i feed_raw.mp4 -vcodec libx264 -crf 23 feed.mp4
scene_data = {'meta': {'fps': fps, 'total_frames': frame_idx - 1, 'resolution': [width, height], 'world_bounds': world_bounds}, 'frames': frames_data}
with open(output_json_path, 'w') as f:
    json.dump(scene_data, f)
